PRODUCER SCRAPER

Import

In [ ]:
import signal as sig
import re #regex

#threading and memory
import threading
import queue

#Data Cleaning
import bs4 as bs 
import requests as req
import pandas as pd
import numpy as np

#Scraping, Selenium and Soup
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

#Camoufox
import camoufox

Settings and Constants


In [4]:
#browser and driver options 
options = Options()
options.binary_location = "C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe"



fresh_href = r"C:\Users\user\Documents\ProgrammingHell\Python\TurboAz scrapper\turboaz_scraper_2026\hrefs\fresh_href.csv"

#websites = []
max_pages = 40
page_count = 0

Functions

In [ ]:
class Scraper:
    def __init__(self,num_consumers):
        
                #--- Constants
        self.main_page = "https://turbo.az"
        self.time_quota = 4
        self.num_consumers = num_consumers
        self.websites = self.producer_link_gen(10)

                #--- Shared state across threads
        self.url_queue = queue.Queue()      #FIFO queue 
        self.seen_urls = set()              #set
        self.seen_lock = threading.Lock()   #thread lock

        self.results = []                   #list for results
        self.results_lock = threading.Lock()# thread safe locking
        
    def create_driver(self):
        '''
        This function supposed to be called by individual threads and creates webdriver instances.
        (Using global driver is not Thread safe!!!)
        '''
        
        service = Service("C:\\chromedriver-win32\\chromedriver.exe")
        options = webdriver.ChromeOptions()
        options.binary_location = "C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe"
        # options.add_argument('--headless')
        # options.add_argument('--no-sandbox')
        # options.add_argument('--disable-dev-shm-usage')
        
        return webdriver.Chrome(options=options, service=service)
    
    
    #crawler and its producer

    def crawler(self, url, driver) -> list:
        '''
        Crawler scraper. Scrapes hrefs,
        creats full link connecting href with
        main page(turbo.az) and sends link to the producer
        '''
        driver.get(url)
        soup = BeautifulSoup(driver.page_source)
    
        return [
            f"{self.main_page}{i['href']}"
            for i in soup.find_all('a', href=True)
            if re.fullmatch(r'/autos/\d+[\w-]*', i['href'])
        ]
    
    
    def producer(self, url):
        """
        Initializes Crawler and redirects links to consumer
        """
        driver = self.create_driver()
        try:
            hrefs = self.crawler(url, driver)
            print(hrefs)
            print(len(hrefs))
            for href in hrefs:
                with self.seen_lock:
                    if href in self.seen_urls:
                        continue
                    self.seen_urls.add(href)
                self.url_queue.put(href)
            print('crawler ended')    
        finally:
            driver.quit()

    def producer_link_gen(self, max_pages):
        for page_num in range(1, max_pages + 1):
            yield f'https://turbo.az/autos?page={page_num}'


    #--- fetcher and its consumer
    #@staticmethod
    def details_parser(self, soup):
        
        rows = soup.select('.product-properties__i')
        properties = {}
        for row in rows:
            name_el = row.select_one('.product-properties__i-name')
            value_el = row.select_one('.product-properties__i-value')
            if name_el == value_el:
                name = name_el.get_text(strip=True)
                value = value_el.get_text(strip=True)
                properties[name] = value
        return properties 

    def fetch_details(self, href, driver):
        """
        Takes parameters from consumer and scrapes whatever i needed from targeted pages(Test)
        """
        driver.get(href)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        properties = self.details_parser(soup)
        return {
        "url": href,
        "city": properties.get("Şəhər"),
        "brand": properties.get("Marka"),
        "model": properties.get("Model"),
        "year": properties.get("Buraxılış ili"),
        "body_type": properties.get("Ban növü"),
        "color": properties.get("Rəng"),
        "engine": properties.get("Mühərrik"),
        "mileage": properties.get("Yürüş"),
    }
    
    
    
    def consumer(self):
        driver = self.create_driver()

        try:
            while True:
                href = self.url_queue.get()
                if href is None:
                    break
                data = self.fetch_details(href, driver)
                print('consumer ended')
                
                with self.results_lock:
                    self.results.append(data)
                self.url_queue.task_done()
                
        finally:
            webdriver.quit()
        

    def run(self):
        # 1. Start consumers FIRST — they'll just block on an empty
        #    queue until producers put work in.
        consumer_threads = [
            threading.Thread(target=self.consumer)
            for _ in range(self.num_consumers)
        ]
        for t in consumer_threads:
            print('comsumer started')
            t.start()
        # for t in consumer_threads:
        #     t.join()

        
        # 2. Start one producer(Crawler starter) thread per website.
        producer_threads = [
            threading.Thread(target=self.producer, args=(url,))
            for url in self.websites
        ]
        for t in producer_threads:
            print('crawler started')
            t.start()
        for t in producer_threads:
            t.join()   # wait until all producers finish finding hrefs



        # 3. Now that no more hrefs are coming, tell each consumer to stop
        for _ in range(self.num_consumers):
            self.url_queue.put(None)

        for t in consumer_threads:
            t.join()

        return self.results        

def main():
    print("Starting web scraping with multithreading...")
    start_time = time.time()
    scraper = Scraper(
        num_consumers= 8
    )
    results = scraper.run()
    print(results)
    end_time = time.time()
    print(f"Scraped {len(results)} listings in {end_time - start_time:.2f} seconds")

  

Activation

In [10]:
if __name__ == "__main__":
    main()

Starting web scraping with multithreading...
comsumer started
comsumer started
comsumer started
comsumer started
comsumer started
comsumer started
comsumer started
comsumer started
crawler started
crawler started
crawler started
crawler started
crawler started
crawler started
crawler started
crawler started
crawler started
crawler started
[]
0
crawler ended
[]
0
crawler ended
['https://turbo.az/autos/9572723-changan-qiyuan-q07', 'https://turbo.az/autos/10408246-ford-transit', 'https://turbo.az/autos/10409169-toyota-land-cruiser', 'https://turbo.az/autos/10511434-geely-galaxy-m9', 'https://turbo.az/autos/10220165-suzuki-sx4', 'https://turbo.az/autos/10233213-kia-niro', 'https://turbo.az/autos/10389183-hyundai-santa-fe', 'https://turbo.az/autos/10404645-kia-sorento', 'https://turbo.az/autos/10501523-land-rover-rr-sport', 'https://turbo.az/autos/10505810-kia-optima', 'https://turbo.az/autos/10512015-porsche-panamera-4s', 'https://turbo.az/autos/10517628-bmw-328', 'https://turbo.az/autos/1

Exception in thread Thread-29 (consumer):
Traceback (most recent call last):
  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 1041, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 992, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Temp\ipykernel_11748\2114702304.py", line 128, in consumer
    webdriver.quit()
    ^^^^^^^^^^^^^^
  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\selenium\webdriver\__init__.py", line 103, in __getattr__
    raise AttributeError(f"module 'selenium.webdriver' has no attribute {name!r}")
AttributeError: module 'selenium.webdriver' has no attribute 'quit'


consumer ended


Exception in thread Thread-33 (consumer):
Traceback (most recent call last):
  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 1041, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 992, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Temp\ipykernel_11748\2114702304.py", line 128, in consumer
    webdriver.quit()
    ^^^^^^^^^^^^^^
  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\selenium\webdriver\__init__.py", line 103, in __getattr__
    raise AttributeError(f"module 'selenium.webdriver' has no attribute {name!r}")
AttributeError: module 'selenium.webdriver' has no attribute 'quit'


consumer ended


...

consumer ended
consumer ended


Traceback (most recent call last):
  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 1041, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 992, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Temp\ipykernel_11748\2114702304.py", line 128, in consumer
    webdriver.quit()
    ^^^^^^^^^^^^^^
  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\selenium\webdriver\__init__.py", line 103, in __getattr__
    raise AttributeError(f"module 'selenium.webdriver' has no attribute {name!r}")
AttributeError: module 'selenium.webdriver' has no attribute 'quit'
Exception in thread Thread-28 (consumer):
Exception in thread Thread-27 (consumer):
Traceback (most recent call last):
Traceback (most recent call last):
  File "c:\Users\user\AppData\Local\Programs\Python\Pyt

consumer ended
consumer ended
consumer ended


Exception in thread Thread-30 (consumer):
Traceback (most recent call last):
  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 1041, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 992, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Temp\ipykernel_11748\2114702304.py", line 128, in consumer
    webdriver.quit()
    ^^^^^^^^^^^^^^
  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\selenium\webdriver\__init__.py", line 103, in __getattr__
    raise AttributeError(f"module 'selenium.webdriver' has no attribute {name!r}")
AttributeError: module 'selenium.webdriver' has no attribute 'quit'


consumer ended
[{'url': 'https://turbo.az/autos/10511434-geely-galaxy-m9', 'city': None, 'brand': None, 'model': None, 'year': None, 'body_type': None, 'color': None, 'engine': None, 'mileage': None}, {'url': 'https://turbo.az/autos/10409169-toyota-land-cruiser', 'city': None, 'brand': None, 'model': None, 'year': None, 'body_type': None, 'color': None, 'engine': None, 'mileage': None}, {'url': 'https://turbo.az/autos/9572723-changan-qiyuan-q07', 'city': None, 'brand': None, 'model': None, 'year': None, 'body_type': None, 'color': None, 'engine': None, 'mileage': None}, {'url': 'https://turbo.az/autos/10408246-ford-transit', 'city': None, 'brand': None, 'model': None, 'year': None, 'body_type': None, 'color': None, 'engine': None, 'mileage': None}, {'url': 'https://turbo.az/autos/10389183-hyundai-santa-fe', 'city': None, 'brand': None, 'model': None, 'year': None, 'body_type': None, 'color': None, 'engine': None, 'mileage': None}, {'url': 'https://turbo.az/autos/10404645-kia-sorento', 

Buffer +

Buffer -

Examples and Ideas